Download [dataset](https://www.kaggle.com/competitions/what-on-the-video)

Place in `data/`

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else torch.device("cpu")

epochs = 10
batch_size = 4
learning_rate = 1e-3
backbone_learning_rate = 1e-5

device

device(type='mps')

In [2]:
from data_reading import TrainDataset, TestDataset
from torch.utils.data import DataLoader
train_dataset = TrainDataset(data_path="data/train", label_path="data/train.csv")
test_dataset = TestDataset(data_path="data/test")

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

num_classes = len(train_dataset[0][1])

In [3]:
from itertools import chain

from transformers import VideoMAEForVideoClassification, VideoMAEImageProcessor

model = VideoMAEForVideoClassification.from_pretrained(
    "MCG-NJU/videomae-base-finetuned-kinetics",
    num_labels=num_classes,
    ignore_mismatched_sizes=True,
)

optimizer = optim.AdamW(
    [
        {"params": model.videomae.parameters(), "lr": backbone_learning_rate},
        {
            "params": chain(model.fc_norm.parameters(), model.classifier.parameters()),
            "lr": learning_rate,
        },
    ],
)
criterion = nn.BCEWithLogitsLoss()



You passed `num_labels=9` which is incompatible to the `id2label` map of length `400`.


Loading weights:   0%|          | 0/162 [00:00<?, ?it/s]

VideoMAEForVideoClassification LOAD REPORT from: MCG-NJU/videomae-base-finetuned-kinetics
Key                                                            | Status     |                                                                                         
---------------------------------------------------------------+------------+-----------------------------------------------------------------------------------------
videomae.encoder.layer.{0...11}.attention.attention.v_bias     | UNEXPECTED |                                                                                         
videomae.encoder.layer.{0...11}.attention.attention.q_bias     | UNEXPECTED |                                                                                         
videomae.encoder.layer.{0...11}.attention.attention.query.bias | MISSING    |                                                                                         
videomae.encoder.layer.{0...11}.attention.attention.key.bias   | MISSING   

In [4]:
from trainer import Trainer

trainer = Trainer(model, criterion, optimizer, device, epoch_amount=epochs)
trainer.fit(train_loader)

Эпоха: 0 

Epoch 0 train:   0%|          | 0/73 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
trainer.save("./model.pt")